# Topic 11 — Practice: Exploratory Analysis & Visualization

**The notebook is the lesson.** Work top to bottom. Cells with `assert` grade themselves — a silent run = pass, an `AssertionError` = fix your code. Warm-Up, Interview Lens and Reflection have **no answer key** — answer in writing.

_Spaced repetition: ~60% of this notebook is the current topic, ~40% revisits earlier topics._

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
products['list_price'] = np.where(products['list_price']<0, np.nan, products['list_price'])
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
orders['order_date'] = pd.to_datetime(orders['order_date'], format='mixed', dayfirst=True, errors='coerce')
orders['month'] = orders['order_date'].dt.to_period('M').astype(str)
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
master = (items.merge(products[['product_id','category','unit_cost']], on='product_id', how='left')
              .merge(orders[['order_id','channel','status','month','order_date']], on='order_id', how='left'))
master['line_cogs'] = master['quantity']*master['unit_cost']
master['line_profit'] = master['line_revenue'] - master['line_cogs']
oi = master  # alias
ts = master.dropna(subset=['order_date']).set_index('order_date').sort_index()
monthly = ts['line_revenue'].resample('ME').sum()


## 🔁 Warm-Up Recall (earlier topics — no answers given)

From **Topics 01–10**:

1. (T10) `pivot` vs `pivot_table` — the crucial difference?
2. (T07) How do you compute month-over-month growth in one call?
3. NumPy: a histogram is built on which NumPy function?

In [ ]:
# NumPy recall: bin counts with np.histogram
data = np.array([1,1,2,3,3,3,4])
counts, edges = ...   # TODO np.histogram(data, bins=4)
assert counts.sum() == len(data)
print(counts)

## 🎯 Core Lesson Tasks (current topic)

Every chart answers a question. Title = the takeaway. Label axes & units. Sort bars.

**Core 1 — trend with takeaway title.** Plot `monthly` with a 3-month moving average overlay; set a takeaway title + ylabel.

In [ ]:
ax = monthly.plot(figsize=(10,4))
# TODO add rolling(3).mean(), a takeaway title, ylabel
assert len(monthly) >= 12
plt.show()

**Core 2 — sorted bar.** Revenue by channel, sorted, labelled, titled.

In [ ]:
channel_rev = master.groupby('channel')['line_revenue'].sum().sort_values(ascending=False)
# TODO bar chart with title + ylabel
assert channel_rev.is_monotonic_decreasing
channel_rev

**Core 3 — distribution.** Histogram of per-order revenue; note skew/outliers in the reflection.

In [ ]:
order_rev = master.groupby('order_id')['line_revenue'].sum()
# TODO hist with title + xlabel
order_rev.describe()

## 🔀 Mixed Review Tasks (earlier topics — spaced repetition)

**Mixed review (Topic 06) — join for the chart.** Build refunds-by-month (join `returns`→order dates, resample).

In [ ]:
returns = pd.read_csv(RAW+'returns.csv', dtype={'order_id':str})
rmonth = returns.merge(orders[['order_id','order_date']], on='order_id', how='left').dropna(subset=['order_date'])
refunds_by_month = ...
assert refunds_by_month.sum() > 0
refunds_by_month.tail()

**Mixed review (Topic 09) — feature then plot.** Profit by `category` (sorted barh).

In [ ]:
cat_profit = ...
assert cat_profit.is_monotonic_increasing or cat_profit.is_monotonic_decreasing
cat_profit

## 🕵️ Data Detective Investigation

**Case file #11 — the one-page dashboard.** Assemble a 2×2 figure for the manager: monthly revenue, revenue by channel, profit by category, profit by channel. This becomes the capstone centerpiece.

In [ ]:
channel_rev = master.groupby('channel')['line_revenue'].sum().sort_values(ascending=False)
fig, axes = plt.subplots(2,2, figsize=(13,8))
monthly.plot(ax=axes[0,0], title='Monthly revenue')
channel_rev.plot(kind='bar', ax=axes[0,1], title='Revenue by channel')
master.groupby('category')['line_profit'].sum().sort_values().plot(kind='barh', ax=axes[1,0], title='Profit by category')
master.groupby('channel')['line_profit'].sum().plot(kind='bar', ax=axes[1,1], title='Profit by channel')
fig.tight_layout()
assert len(fig.axes) >= 4
plt.show()

## 🔎 Interview Lens (write answers — no model answers)

- **Q30:** A stakeholder doubts your number — how do you trace it to raw data and defend/correct it?
- **Q31:** How do you make the analysis reproducible next quarter?

## ✍️ Reflection (write short explanations)

1. Answer Q30 and Q31 in writing.
2. Which single chart best explains the case to a manager?
3. **Investigation log:** finalize the story for the capstone.